# Zomato Bangalore — Restaurant Analytics
**Dataset:** Zomato Kaggle (Bangalore) · **Rows after cleaning:** 49,011 · **Author:** Yashvardhan Debas

---
Exploratory analysis of Bangalore's restaurant ecosystem — covering pricing, ratings, cuisine patterns,
location performance, and market opportunity gaps.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── plot defaults ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
PALETTE = 'YlOrRd'

df = pd.read_csv(r'zomato.csv', encoding='latin-1')
print(f'Raw dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 2. Data Cleaning

In [ ]:
# ── numeric cleaning ───────────────────────────────────────────────────────────
def clean_numeric(col):
    cleaned = col.astype(str).str.replace('/5', '', regex=False).str.replace(',', '', regex=False).str.strip()
    numeric = pd.to_numeric(cleaned, errors='coerce')
    return numeric.fillna(numeric.mean()).round(2)

for col in ['rate', 'approx_cost(for two people)']:
    df[col] = clean_numeric(df[col])

df['phone'] = df['phone'].fillna('Unknown').astype(str).str.strip()

# ── drop high-null columns ─────────────────────────────────────────────────────
df.drop(columns=['dish_liked', 'menu_item'], inplace=True)

# ── fix reviews_list ───────────────────────────────────────────────────────────
df['reviews_list'] = df['reviews_list'].replace('[]', np.nan)

# ── drop corrupted url rows ────────────────────────────────────────────────────
review_keywords = 'Rated|RATED|food|delicious|parking|ordered|staff|taste|worth'
df = df[~df['url'].str.contains(review_keywords, case=False, na=False)].copy()

# ── drop remaining sparse nulls ────────────────────────────────────────────────
df.dropna(subset=['location', 'rest_type', 'cuisines'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Null values remaining:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

## 3. Univariate Analysis

In [ ]:
# Q1 — rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['rate'], bins=25, kde=True, color='#E05C2A', ax=axes[0])
axes[0].axvline(df['rate'].mean(), color='#333', linestyle='--', lw=1.5, label=f"Mean: {df['rate'].mean():.2f}")
axes[0].axvline(df['rate'].median(), color='#888', linestyle=':', lw=1.5, label=f"Median: {df['rate'].median():.2f}")
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].legend(frameon=False)

# Q2 — cost distribution
cost_capped = df[df['approx_cost(for two people)'] <= 3000]['approx_cost(for two people)']
sns.histplot(cost_capped, bins=30, kde=True, color='#F5A623', ax=axes[1])
axes[1].axvline(cost_capped.median(), color='#333', linestyle='--', lw=1.5, label=f"Median: ₹{cost_capped.median():.0f}")
axes[1].set_title('Approx Cost for Two (capped at ₹3,000)')
axes[1].set_xlabel('Cost (INR)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{int(x):,}'))
axes[1].legend(frameon=False)

plt.suptitle('Distributions — Rating & Cost', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(df[['rate', 'approx_cost(for two people)']].describe().round(2))

In [ ]:
# Q3 — restaurant type breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_rest = df['rest_type'].value_counts().head(10)
colors = sns.color_palette('YlOrRd_r', len(top_rest))
axes[0].barh(top_rest.index[::-1], top_rest.values[::-1], color=colors[::-1])
axes[0].set_title('Top 10 Restaurant Types')
axes[0].set_xlabel('Count')
for i, v in enumerate(top_rest.values[::-1]):
    axes[0].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

# Q6 — listed_in type
listed = df['listed_in(type)'].value_counts()
axes[1].barh(listed.index[::-1], listed.values[::-1], color=sns.color_palette('YlOrRd_r', len(listed))[::-1])
axes[1].set_title('Restaurants by Category')
axes[1].set_xlabel('Count')
for i, v in enumerate(listed.values[::-1]):
    axes[1].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

plt.suptitle('Restaurant Type & Category Breakdown', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Bivariate Analysis

In [ ]:
# Q7 & Q8 — service features vs rating
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, title in zip(axes,
                           ['online_order', 'book_table'],
                           ['Online Ordering vs Rating', 'Table Booking vs Rating']):
    means = df.groupby(col)['rate'].mean().round(2)
    bars = ax.bar(means.index, means.values,
                  color=['#F5A623', '#E05C2A'], width=0.4, edgecolor='white')
    ax.set_ylim(3.4, 4.0)
    ax.set_title(title)
    ax.set_ylabel('Average Rating')
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005,
                f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Do Service Features Affect Ratings?', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# print summary
for col in ['online_order', 'book_table']:
    print(f'\n{col}:')
    print(df.groupby(col)['rate'].agg(['mean', 'count']).round(2))

In [ ]:
# Q9 — price range vs votes
df['cost_bucket'] = pd.cut(
    df['approx_cost(for two people)'],
    bins=[0, 200, 500, 1000, 2000, 10000],
    labels=['Budget\n(₹0-200)', 'Affordable\n(₹200-500)',
            'Mid-range\n(₹500-1k)', 'Premium\n(₹1k-2k)', 'Luxury\n(₹2k+)']
)

bucket_stats = df.groupby('cost_bucket', observed=True).agg(
    avg_votes=('votes', 'mean'),
    count=('name', 'count')
).round(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = sns.color_palette('YlOrRd', len(bucket_stats))
axes[0].bar(bucket_stats.index, bucket_stats['avg_votes'], color=colors, edgecolor='white')
axes[0].set_title('Average Votes by Price Range')
axes[0].set_ylabel('Average Votes')
for i, v in enumerate(bucket_stats['avg_votes']):
    axes[0].text(i, v + 2, f'{v:.0f}', ha='center', fontsize=9, fontweight='bold')

axes[1].bar(bucket_stats.index, bucket_stats['count'], color=colors, edgecolor='white')
axes[1].set_title('Restaurant Count by Price Range')
axes[1].set_ylabel('Number of Restaurants')
for i, v in enumerate(bucket_stats['count']):
    axes[1].text(i, v + 30, f'{v:,.0f}', ha='center', fontsize=9)

plt.suptitle('Price Range — Popularity vs Supply', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Q10 & Q11 — location: count and rating
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

top_locs_count = df['location'].value_counts().head(15)
axes[0].barh(top_locs_count.index[::-1], top_locs_count.values[::-1],
             color=sns.color_palette('YlOrRd_r', 15)[::-1])
axes[0].set_title('Top 15 Locations by Restaurant Count')
axes[0].set_xlabel('Number of Restaurants')

top_locs_rating = df.groupby('location')['rate'].mean().sort_values(ascending=False).head(15).round(2)
axes[1].barh(top_locs_rating.index[::-1], top_locs_rating.values[::-1],
             color=sns.color_palette('YlOrRd_r', 15)[::-1])
axes[1].set_title('Top 15 Locations by Average Rating')
axes[1].set_xlabel('Average Rating')
axes[1].set_xlim(3.7, 4.3)
for i, v in enumerate(top_locs_rating.values[::-1]):
    axes[1].text(v + 0.005, i, f'{v}', va='center', fontsize=9, fontweight='bold')

plt.suptitle('Location Analysis — Volume vs Quality', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Multivariate Analysis

In [ ]:
# cuisine explode — used for Q19 onwards
df_exploded = df.copy()
df_exploded['cuisines'] = df_exploded['cuisines'].str.split(', ')
df_exploded = df_exploded.explode('cuisines')
df_exploded['cuisines'] = df_exploded['cuisines'].str.strip()
print(f'Exploded shape: {df_exploded.shape[0]:,} rows (one row per cuisine per restaurant)')
print(f"\nTop 10 cuisines by restaurant count:")
print(df_exploded['cuisines'].value_counts().head(10).to_string())

In [ ]:
# Q19 — top rated cuisines by location
top_locs = ['Lavelle Road', 'St. Marks Road', 'Koramangala 5th Block',
            'Koramangala 3rd Block', 'Church Street']

cuisine_loc = (
    df_exploded[df_exploded['location'].isin(top_locs)]
    .groupby(['location', 'cuisines'])['rate']
    .mean().round(2).reset_index()
)
top_per_loc = cuisine_loc.sort_values('rate', ascending=False).groupby('location').head(3)

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=top_per_loc, x='location', y='rate', hue='cuisines',
            palette='YlOrRd', ax=ax)
ax.set_ylim(3.8, 5.0)
ax.set_title('Top 3 Rated Cuisines per High-Performing Location')
ax.set_xlabel('')
ax.set_ylabel('Average Rating')
ax.legend(title='Cuisine', bbox_to_anchor=(1.01, 1), borderaxespad=0, frameon=False)
plt.tight_layout()
plt.show()

# insight
print("""
INSIGHT — Niche international cuisines (Japanese, Korean, Mediterranean)
consistently outrate common cuisines across Bangalore's top locations.
Specialisation correlates strongly with higher customer satisfaction.
""")

In [ ]:
# Q20 — cuisine identity per rest_type
target_types = ['Cafe', 'Casual Dining', 'Delivery', 'Quick Bites']

cuisine_resttype = (
    df_exploded[df_exploded['rest_type'].isin(target_types)]
    .groupby(['rest_type', 'cuisines'])['votes']
    .sum().reset_index()
)
top_5 = cuisine_resttype.sort_values('votes', ascending=False).groupby('rest_type').head(5)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes = axes.flatten()
colors = ['#E05C2A', '#F5A623', '#FDD27E', '#FDEFC3', '#FFF8E7']

for i, rtype in enumerate(target_types):
    data = top_5[top_5['rest_type'] == rtype].sort_values('votes', ascending=True)
    axes[i].barh(data['cuisines'], data['votes'], color=colors[:len(data)][::-1])
    axes[i].set_title(rtype, fontsize=13)
    axes[i].set_xlabel('Total Votes')
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))

plt.suptitle('Top 5 Cuisines by Votes — Per Restaurant Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
INSIGHT — Each restaurant type has a distinct cuisine identity:
  Cafes       → Western (Burger, Continental, Italian, American)
  Casual Dining → Indian (North Indian dominates at 2.6M votes)
  Quick Bites → Mixed, North Indian + Fast Food + Chinese
  Delivery    → North Indian leads but 20x fewer votes than Casual Dining
""")

In [ ]:
# Q21 — category distribution by location (excluding Delivery)
listed_no_delivery = (
    df[df['listed_in(type)'] != 'Delivery']
    .groupby(['location', 'listed_in(type)'])['name']
    .count().reset_index()
)
listed_no_delivery.columns = ['location', 'listed_type', 'count']

pivot = listed_no_delivery.pivot_table(
    index='location', columns='listed_type', values='count', fill_value=0
)
top_15 = df['location'].value_counts().head(15).index
pivot_f = pivot.loc[pivot.index.isin(top_15)]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot_f, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.4, linecolor='white', ax=ax,
            cbar_kws={'label': 'Restaurant Count'})
ax.set_title('Restaurant Category Distribution by Location (excl. Delivery)', pad=15)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Q22 — price range vs votes by location
cost_loc = (
    df.groupby(['location', 'cost_bucket'], observed=True)['votes']
    .mean().round(0).reset_index()
)
top_bucket = cost_loc.sort_values('votes', ascending=False).groupby('location').head(1)
top_15_by_votes = top_bucket.sort_values('votes', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(13, 7))
palette = {'Budget\n(₹0-200)': '#FFF0C4', 'Affordable\n(₹200-500)': '#FDD27E',
           'Mid-range\n(₹500-1k)': '#F5A623', 'Premium\n(₹1k-2k)': '#E05C2A',
           'Luxury\n(₹2k+)': '#8B1A00'}
sns.barplot(data=top_15_by_votes, x='votes', y='location',
            hue='cost_bucket', palette=palette, dodge=False, ax=ax)
ax.set_title('Highest Voted Price Range per Location (Top 15)')
ax.set_xlabel('Average Votes')
ax.set_ylabel('')
ax.legend(title='Price Range', bbox_to_anchor=(1.01, 1), frameon=False)
plt.tight_layout()
plt.show()

print("""
INSIGHT — Budget restaurants are absent from the top-voted list entirely.
Luxury (₹2k+) drives highest engagement in St. Marks Road (5,277 avg votes).
Premium (₹1k-2k) dominates Koramangala, Indiranagar, and Brigade Road corridors.
""")

In [ ]:
# Q23 — underserved area analysis
location_stats = df.groupby('location').agg(
    restaurant_count=('name', 'count'),
    avg_votes=('votes', 'mean'),
    avg_rating=('rate', 'mean')
).round(2).reset_index()

location_stats['demand_supply_ratio'] = (
    location_stats['avg_votes'] / location_stats['restaurant_count']
).round(2)

underserved = location_stats[
    (location_stats['demand_supply_ratio'] > 10) &
    (location_stats['restaurant_count'] < 50)
].sort_values('demand_supply_ratio', ascending=False)

# scatter plot
fig, ax = plt.subplots(figsize=(13, 7))
scatter = ax.scatter(
    location_stats['restaurant_count'],
    location_stats['avg_votes'],
    s=location_stats['demand_supply_ratio'] * 6,
    c=location_stats['demand_supply_ratio'],
    cmap='YlOrRd', alpha=0.65, edgecolors='white', linewidth=0.5
)
plt.colorbar(scatter, ax=ax, label='Demand / Supply Ratio')

for _, row in underserved.iterrows():
    ax.annotate(
        row['location'],
        xy=(row['restaurant_count'], row['avg_votes']),
        xytext=(12, 4), textcoords='offset points',
        fontsize=9, fontweight='bold', color='#8B1A00',
        arrowprops=dict(arrowstyle='->', color='#8B1A00', lw=0.8)
    )

ax.set_title('Restaurant Supply vs Demand by Location\nTop-left = underserved (high demand, low supply)', pad=12)
ax.set_xlabel('Number of Restaurants (Supply)')
ax.set_ylabel('Average Votes (Demand)')
plt.tight_layout()
plt.show()

# missing cuisines
citywide_top = df_exploded['cuisines'].value_counts().head(10).index.tolist()
print('Underserved locations and their missing top cuisines:\n')
for _, row in underserved.iterrows():
    loc = row['location']
    existing = df_exploded[df_exploded['location'] == loc]['cuisines'].unique().tolist()
    missing = [c for c in citywide_top if c not in existing]
    print(f"  {loc} (ratio {row['demand_supply_ratio']:.0f}, {int(row['restaurant_count'])} restaurants)")
    print(f"  Missing: {', '.join(missing) if missing else 'None'}\n")

## 6. Key Findings

| # | Finding | Implication |
|---|---------|-------------|
| 1 | Niche international cuisines rate higher than common ones | Specialisation drives satisfaction |
| 2 | Casual Dining generates 20× more votes than Delivery | Dine-in drives far greater engagement |
| 3 | Cafes are Western-food dominated; Casual Dining is Indian | Distinct identities per format |
| 4 | Luxury (₹2k+) restaurants receive the most votes | Higher spend = higher engagement |
| 5 | Rajarajeshwari Nagar: ratio 183, only 2 restaurants | Highest opportunity gap in Bangalore |
| 6 | South Indian, Cafe, Desserts missing from all underserved zones | Universal gaps for new entrants |